In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

In [2]:
# Load dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Normalize pixel values (0–255 → 0–1)
x_train = x_train / 255.0
x_test = x_test / 255.0

# Reshape to add channel dimension (28x28 → 28x28x1)
x_train = x_train.reshape(-1, 28, 28, 1)
x_test = x_test.reshape(-1, 28, 28, 1)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [3]:
model = models.Sequential([

    # Convolution Layer 1
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)),
    layers.MaxPooling2D((2,2)),

    # Convolution Layer 2
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    # Flatten
    layers.Flatten(),

    # Fully Connected Layers
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')  # 10 classes (digits 0–9)
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [4]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [5]:
history = model.fit(
    x_train, y_train,
    epochs=5,
    validation_data=(x_test, y_test)
)

Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 54s 28ms/step - accuracy: 0.9588 - loss: 0.1376 - val_accuracy: 0.9831 - val_loss: 0.0484
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 52s 28ms/step - accuracy: 0.9855 - loss: 0.0457 - val_accuracy: 0.9880 - val_loss: 0.0368
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 81s 27ms/step - accuracy: 0.9908 - loss: 0.0306 - val_accuracy: 0.9897 - val_loss: 0.0308
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 51s 27ms/step - accuracy: 0.9927 - loss: 0.0229 - val_accuracy: 0.9911 - val_loss: 0.0290
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 51s 27ms/step - accuracy: 0.9944 - loss: 0.0174 - val_accuracy: 0.9905 - val_loss: 0.0298


In [6]:
test_loss, test_acc = model.evaluate(x_test, y_test)
print("Test Accuracy:", test_acc)

313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.9905 - loss: 0.0298
Test Accuracy: 0.9904999732971191


In [7]:
predictions = model.predict(x_test)

# Example: first image
import numpy as np
print("Predicted:", np.argmax(predictions[0]))
print("Actual:", y_test[0])

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step
Predicted: 7
Actual: 7


In [8]:
# Print model summary
model.summary()

# Print total parameters
print("Total parameters:", model.count_params())

# Print trainable and non-trainable parameters separately
trainable_params = sum([w.shape.num_elements() for w in model.trainable_weights])
non_trainable_params = sum([w.shape.num_elements() for w in model.non_trainable_weights])

print("Trainable parameters:", trainable_params)
print("Non-trainable parameters:", non_trainable_params)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │       102,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 365,792 (1.40 MB)

 Trainable params: 121,930 (476.29 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 243,862 (952.59 KB)

Total parameters: 121930
Trainable parameters: 121930
Non-trainable parameters: 0


In [9]:
for layer in model.layers:
    if hasattr(layer, 'kernel'): # Weights
        print(f"\nLayer: {layer.name}")
        print(f"  Weights (kernel):\n{layer.kernel}")
    if hasattr(layer, 'bias'): # Biases
        print(f"  Biases:\n{layer.bias}")


Layer: conv2d
  Weights (kernel):
<Variable path=sequential/conv2d/kernel, shape=(3, 3, 1, 32), dtype=float32, value=[[[[-6.88241944e-02 -2.65386909e-01 -7.45223984e-02  1.86926305e-01
     4.47260976e-01  1.95794567e-01 -1.57891795e-01  2.09624767e-02
    -1.66416913e-01 -6.26273314e-03 -2.37774909e-01  2.22578526e-01
     1.94027007e-01  3.60800931e-03  1.63084596e-01  3.47545594e-01
    -4.16966993e-03  5.45213968e-02 -3.30885708e-01  1.98902607e-01
    -1.72063425e-01 -1.70514345e-01 -4.28400747e-02  1.64705500e-01
    -3.72334212e-01  8.03773776e-02 -3.58954221e-02  6.38218224e-02
    -3.02818660e-02  2.52470940e-01 -7.38795847e-02  6.84298053e-02]]

  [[ 4.78824191e-02 -3.74175310e-01 -1.09349653e-01  1.57134712e-01
    -1.18136659e-01  1.73292711e-01 -2.81148590e-02 -1.31799638e-01
    -2.79572189e-01  2.72077411e-01  3.74227464e-02  2.91405827e-01
    -2.44270638e-02  7.82435238e-02  2.07939595e-01  1.57492086e-01
     7.49251768e-02  9.57663283e-02  4.57642376e-02 -8.77634510